# Part 0 — chạy trên Google Colab với CUDA
Chọn **Runtime → Change runtime type → T4 GPU → Save**, rồi **Run all**.
Notebook chứa phiên bản Part 0 đã hoàn thiện trong checkout; tải dữ liệu và script từ repo GitHub. Không chạy Part 1–4.
Chỉ khi output có `Device: cuda` và dòng `PASS` cuối cùng mới xác nhận chạy GPU thành công.


In [ ]:
# Colab: Runtime > Change runtime type > T4 GPU, sau đó Run all.
import os, subprocess, sys, json
from pathlib import Path
import torch

print("PyTorch:", torch.__version__)
assert torch.cuda.is_available(), "Chưa có CUDA. Chọn Runtime > Change runtime type > T4 GPU rồi chạy lại."
print("CUDA GPU:", torch.cuda.get_device_name(0))
root = Path("/content/K4-Track4-Day1-Neuralnetwork")
if not root.exists():
    subprocess.run(["git", "clone", "https://github.com/namhv521/K4-Track4-Day1-Neuralnetwork", str(root)], check=True)
assert (root / "data/covtype.csv.gz").is_file(), "Repo thiếu dữ liệu gốc"
assert (root / "data/split_metadata.csv").is_file(), "Repo thiếu metadata"
dest = root / "submission_2A202602853/code"
dest.mkdir(parents=True, exist_ok=True)
sources = {'data.py': '"""data.py — Chuẩn bị dữ liệu cho Part 0.\n\nNhiệm vụ: nạp tập train/eval đã chia sẵn, tách validation từ train, chuẩn hoá, đưa lên thiết bị.\n\nĐiều kiện trước: đã chạy `python scripts/split_data.py` (tạo data/processed/train.npz, eval.npz).\n\nQuy ước dữ liệu (xem README mục 2 và 3):\n    X : float32, shape (N, 54)   — 10 cột đầu là số liên tục, 44 cột sau là nhị phân (one-hot)\n    y : int64,   shape (N,)      — nhãn 0..6\nTập eval CHỈ dùng để chấm điểm cuối. Không dùng nó để chọn cấu hình, chuẩn hoá hay dừng sớm.\n"""\nfrom __future__ import annotations\n\nimport numpy as np\nimport torch\nfrom pathlib import Path\nfrom sklearn.model_selection import train_test_split\n\nN_NUMERIC = 10  # số cột liên tục cần chuẩn hoá (cột 0..9)\n\n\ndef load_split(processed_dir: str = "data/processed"):\n    """Nạp train và eval từ file .npz.\n\n    Trả về: X_train_full, y_train_full, X_eval, y_eval, eval_row_id\n    Các bước:\n      1. np.load(f"{processed_dir}/train.npz") -> khoá "X", "y"\n      2. np.load(f"{processed_dir}/eval.npz")  -> khoá "X", "y", "row_id"\n      3. assert shape/dtype đúng quy ước ở đầu file\n    """\n    root = Path(processed_dir)\n    with np.load(root / "train.npz") as train, np.load(root / "eval.npz") as ev:\n        X_train, y_train = train["X"], train["y"]\n        X_eval, y_eval, row_id = ev["X"], ev["y"], ev["row_id"]\n    for X, y in ((X_train, y_train), (X_eval, y_eval)):\n        assert X.ndim == 2 and X.shape[1] == 54 and X.dtype == np.float32\n        assert y.shape == (len(X),) and y.dtype == np.int64\n        assert np.isfinite(X).all() and ((y >= 0) & (y < 7)).all()\n    assert row_id.shape == y_eval.shape and row_id.dtype == np.int64\n    assert len(np.unique(row_id)) == len(row_id)\n    return X_train, y_train, X_eval, y_eval, row_id\n\n\ndef make_val_split(X, y, val_fraction: float = 0.2, seed: int = 42):\n    """Tách validation TỪ train (không đụng eval). Phân tầng theo nhãn.\n\n    Trả về: X_tr, y_tr, X_val, y_val\n    Gợi ý: sklearn.model_selection.train_test_split(..., stratify=y, random_state=seed)\n    Dùng CÙNG seed và val_fraction cho mọi thí nghiệm để so sánh công bằng.\n    """\n    X_tr, X_val, y_tr, y_val = train_test_split(\n        X, y, test_size=val_fraction, stratify=y, random_state=seed\n    )\n    return X_tr, y_tr, X_val, y_val\n\n\ndef fit_standardizer(X_tr):\n    """Tính mean và std của N_NUMERIC cột đầu CHỈ trên tập train (sau khi tách val).\n\n    Trả về: mean (shape (10,)), std (shape (10,))\n    Câu hỏi: vì sao không được tính trên toàn bộ dữ liệu hay trên eval?\n    """\n    # Float64 accumulation avoids rounding error on hundreds of thousands of rows.\n    numeric = X_tr[:, :N_NUMERIC]\n    mean = numeric.mean(axis=0, dtype=np.float64)\n    std = numeric.std(axis=0, dtype=np.float64)\n    std[std == 0] = 1.0\n    return mean, std\n\n\ndef apply_standardizer(X, mean, std):\n    """Trả về bản sao của X, trong đó 10 cột đầu được (x - mean) / std; 44 cột nhị phân giữ nguyên.\n\n    Chú ý: không sửa X tại chỗ nếu bạn còn dùng lại nó; chú ý std = 0 (nếu có).\n    """\n    scaled = X.astype(np.float32, copy=True)\n    safe_std = np.where(std == 0, 1.0, std)\n    scaled[:, :N_NUMERIC] = (X[:, :N_NUMERIC] - mean) / safe_std\n    return scaled\n\n\ndef prepare_data(device: str, val_fraction: float = 0.2, seed: int = 42,\n                 processed_dir: str = "data/processed") -> dict:\n    """Gộp các bước trên và đưa TOÀN BỘ dữ liệu lên `device` một lần (không dùng DataLoader).\n\n    Trả về dict gồm các tensor trên device:\n        X_tr, y_tr, X_val, y_val, X_eval, y_eval        (y là int64)\n    và các mảng numpy: eval_row_id\n    Các bước:\n      1. load_split -> make_val_split -> fit_standardizer (chỉ trên X_tr)\n      2. apply_standardizer cho X_tr, X_val, X_eval bằng CÙNG mean/std\n      3. torch.tensor(..., device=device); X là float32, y là int64\n      4. in ra kích thước các tập và accuracy của chiến lược "luôn đoán lớp đa số" trên val\n    """\n    X_full, y_full, X_eval, y_eval, row_id = load_split(processed_dir)\n    X_tr, y_tr, X_val, y_val = make_val_split(X_full, y_full, val_fraction, seed)\n    mean, std = fit_standardizer(X_tr)\n    result = {"eval_row_id": row_id, "mean": mean, "std": std}\n    for name, X, y in (("tr", X_tr, y_tr), ("val", X_val, y_val),\n                       ("eval", X_eval, y_eval)):\n        result[f"X_{name}"] = torch.as_tensor(\n            apply_standardizer(X, mean, std), dtype=torch.float32, device=device\n        )\n        result[f"y_{name}"] = torch.as_tensor(y, dtype=torch.int64, device=device)\n        print(f"{name}: X={tuple(X.shape)}, y={tuple(y.shape)}, device={device}")\n    majority = np.bincount(y_tr, minlength=7).argmax()\n    print(f"Majority class = {majority}; val accuracy = {(y_val == majority).mean():.6f}")\n    return result\n\n\ndef iterate_batches(X, y, batch_size: int, generator: torch.Generator | None = None, shuffle: bool = True):\n    """Generator trả về từng cặp (xb, yb), thay cho DataLoader.\n\n    Các bước:\n      1. nếu shuffle: perm = torch.randperm(len(X), generator=generator, device=X.device); ngược lại arange\n      2. for i in range(0, N, batch_size): idx = perm[i:i+batch_size]; yield X[idx], y[idx]\n    Chú ý: batch cuối có thể nhỏ hơn batch_size; hãy quyết định bạn xử lý thế nào và ghi lại.\n    """\n    if batch_size <= 0:\n        raise ValueError("batch_size must be positive")\n    if len(X) != len(y):\n        raise ValueError("X and y must contain the same number of samples")\n    if shuffle:\n        indices = torch.randperm(len(X), generator=generator, device=X.device)\n    else:\n        indices = torch.arange(len(X), device=X.device)\n    # Include the final partial batch: no samples are discarded.\n    for start in range(0, len(X), batch_size):\n        idx = indices[start:start + batch_size]\n        yield X[idx], y[idx]\n', 'model.py': '"""model.py — PSEUDO-CODE. Bạn phải tự hoàn thiện mọi hàm/class có `raise NotImplementedError`.\n\nModel: MLP cho bài toán 7 lớp, shape cố định (xem README mục 3 và GUIDE, "Quy định kiến trúc"):\n\n    x (B, 54) -> Linear(54, h1) -> ReLU -> [Dropout] -> Linear(h1, h2) -> ReLU -> [Dropout]\n              -> ... -> Linear(h_last, 7) -> logits (B, 7)\n\nQuy tắc:\n  - Lớp cuối ra logit thô, KHÔNG softmax trong model (softmax nằm trong hàm mất mát).\n  - Dropout chỉ đặt sau ReLU của lớp ẩn; không đặt trên đầu vào hay logit.\n  - Mọi nn.Linear đều có bias. Không BatchNorm, không residual.\n  - Số tham số phải khớp EXPECTED_PARAMS bên dưới.\n"""\nfrom __future__ import annotations\n\nimport torch\nimport torch.nn as nn\n\n# Số tham số bắt buộc ứng với từng kiến trúc (in_features=54, num_classes=7)\nEXPECTED_PARAMS = {\n    (256, 128): 47_879,        # M-base  (baseline)\n    (512, 256): 161_287,       # M-wide  (tuỳ chọn)\n    (256, 128, 64): 55_687,    # M-deep  (tuỳ chọn)\n}\n\n\nclass MLP(nn.Module):\n    """MLP theo quy định ở đầu file.\n\n    Args:\n        hidden:   tuple số nơ-ron các lớp ẩn, ví dụ (256, 128)\n        dropout:  xác suất TẮT nơ-ron q (nn.Dropout dùng p chính là xác suất tắt); 0.0 = không dùng\n        init:     "zeros" | "normal" | "xavier" | "he" | "default"\n    """\n\n    def __init__(self, hidden=(256, 128), dropout: float = 0.0, init: str = "he",\n                 in_features: int = 54, num_classes: int = 7):\n        super().__init__()\n        # TODO các bước:\n        #   1. dựng danh sách lớp: với mỗi h trong hidden: Linear(in, h), ReLU, Dropout(dropout)\n        #   2. thêm Linear(h_cuối, num_classes) làm lớp ra\n        #   3. gộp bằng nn.Sequential (hoặc tự viết forward), lưu vào self.net\n        #   4. gọi init_weights(self, init)\n        raise NotImplementedError\n\n    def forward(self, x: torch.Tensor) -> torch.Tensor:\n        """x: (B, 54) float32  ->  logits: (B, 7) float32."""\n        raise NotImplementedError  # TODO\n\n\ndef init_weights(model: nn.Module, init: str) -> None:\n    """Khởi tạo tham số của MỌI nn.Linear (bias luôn = 0).\n\n    init:\n        "zeros"   : W = 0\n        "normal"  : W ~ N(0, 0.01^2)\n        "xavier"  : nn.init.xavier_normal_ (Var = 2/(n_in+n_out)); nếu bạn dùng Var = 1/n_in theo slide, hãy ghi rõ\n        "he"      : nn.init.kaiming_normal_(w, nonlinearity="relu")  (Var = 2/n_in)\n        "default" : không làm gì (giữ khởi tạo mặc định của nn.Linear; KHÔNG phải He)\n    Gợi ý: duyệt model.modules(), chọn isinstance(m, nn.Linear).\n    """\n    raise NotImplementedError  # TODO\n\n\ndef count_params(model: nn.Module) -> int:\n    """Tổng số tham số huấn luyện được. Dùng để assert với EXPECTED_PARAMS ngay sau khi tạo model."""\n    raise NotImplementedError  # TODO\n\n\n@torch.no_grad()\ndef activation_stats(model: nn.Module, x: torch.Tensor) -> list[float]:\n    """Độ lệch chuẩn của kích hoạt sau mỗi lớp (ở bước 0, một lô val) — dùng cho thí nghiệm khởi tạo.\n\n    Các bước:\n      1. model.eval(); h = x\n      2. duyệt từng lớp con theo thứ tự; sau mỗi nn.Linear (hoặc sau mỗi ReLU, bạn chọn và ghi rõ) lưu h.std().item()\n      3. trả về danh sách std theo lớp\n    """\n    raise NotImplementedError  # TODO\n', 'optimizer.py': '"""optimizer.py — PSEUDO-CODE. Bạn phải tự hoàn thiện mọi hàm có `raise NotImplementedError`.\n\nĐược dùng torch.optim.* và torch.nn.utils.clip_grad_norm_ (xem README mục 5).\nFile này gom việc chọn bộ tối ưu và cắt gradient để `train.py` gọn và mọi thí nghiệm công bằng.\n\nCông thức cần hiểu (slide Chương 4):\n    SGD            : w <- w - lr * g\n    SGD + momentum : v <- mu * v + g ;  w <- w - lr * v          (dạng PyTorch)\n    Adam           : m <- b1 m + (1-b1) g ; v <- b2 v + (1-b2) g^2 ; w <- w - lr * m_hat / (sqrt(v_hat) + eps)\n    AdamW          : như Adam nhưng suy giảm trọng số tách riêng: w <- w - lr * wd * w - lr * m_hat / (sqrt(v_hat) + eps)\n"""\nfrom __future__ import annotations\n\nimport torch\n\nOPTIMIZERS = ("sgd", "sgd_momentum", "adam", "adamw")\n\n\ndef build_optimizer(name: str, params, lr: float, weight_decay: float = 0.0,\n                    momentum: float = 0.9, betas=(0.9, 0.999), eps: float = 1e-8):\n    """Trả về một torch.optim.Optimizer.\n\n    Các bước:\n      1. kiểm tra name nằm trong OPTIMIZERS, nếu không raise ValueError\n      2. "sgd"          -> torch.optim.SGD(params, lr=lr, weight_decay=weight_decay)\n         "sgd_momentum" -> torch.optim.SGD(params, lr=lr, momentum=momentum, weight_decay=weight_decay)\n         "adam"         -> torch.optim.Adam(params, lr=lr, betas=betas, eps=eps, weight_decay=weight_decay)\n         "adamw"        -> torch.optim.AdamW(params, lr=lr, betas=betas, eps=eps, weight_decay=weight_decay)\n    Chú ý: weight_decay của Adam (L2 trộn vào gradient) khác weight_decay của AdamW (suy giảm tách riêng).\n    """\n    raise NotImplementedError  # TODO\n\n\ndef build_scheduler(optimizer, name: str | None, total_steps: int, **kwargs):\n    """(Tuỳ chọn) Bộ lập lịch tốc độ học, ví dụ cosine (slide có ví dụ CosineAnnealingLR).\n\n    Trả về None nếu name là None. Nếu bạn dùng scheduler ở một thí nghiệm, hãy ghi vào bảng (cột notes).\n    """\n    raise NotImplementedError  # TODO\n\n\ndef clip_gradients(params, max_norm: float | None) -> float:\n    """Cắt gradient theo chuẩn L2 toàn cục, và TRẢ VỀ chuẩn gradient TRƯỚC KHI cắt.\n\n    Các bước:\n      1. nếu max_norm là None: tính chuẩn toàn cục mà không cắt (ví dụ clip_grad_norm_ với max_norm=inf)\n      2. ngược lại: total_norm = torch.nn.utils.clip_grad_norm_(params, max_norm)\n      3. return float(total_norm)\n    Giá trị trả về chính là `grad_norm` bạn phải ghi lại ở mỗi bước (để thấy "gai" gradient).\n    Khi dùng mixed precision FP16 + GradScaler: phải scaler.unscale_(optimizer) TRƯỚC khi gọi hàm này.\n    """\n    raise NotImplementedError  # TODO\n', 'plots.py': '"""plots.py — PSEUDO-CODE. Bạn phải tự hoàn thiện mọi hàm có `raise NotImplementedError`.\n\nẢnh biểu đồ là sản phẩm nộp (xem README mục 6): mỗi thí nghiệm một ảnh figures/<exp_id>.png.\nKhi notebook chạy trong code/, lưu vào "../figures/" (ví dụ path = f"../figures/{exp_id}.png").\n"""\nfrom __future__ import annotations\n\nimport matplotlib.pyplot as plt\n\n\ndef plot_run(result: dict, path: str) -> None:\n    """Vẽ MỘT thí nghiệm thành một ảnh PNG có ít nhất 3 ô:\n         (1) train_loss và val_loss theo epoch (cùng một trục)\n         (2) val_acc (và nên có val_macro_f1) theo epoch\n         (3) grad_norm theo epoch (đo TRƯỚC khi clip)\n    Yêu cầu: tiêu đề ghi exp_id và cấu hình chính (optimizer, lr, batch, ...), có nhãn trục và chú thích.\n    Các bước: fig, axes = plt.subplots(1, 3, figsize=...); plot; set_title/xlabel/legend;\n              fig.savefig(path, dpi=..., bbox_inches="tight"); plt.close(fig)\n    Gợi ý: đánh dấu best_epoch bằng đường thẳng đứng.\n    """\n    raise NotImplementedError  # TODO\n\n\ndef plot_compare(results: list[dict], metric: str, path: str, title: str = "") -> None:\n    """Vẽ chồng một chỉ số (ví dụ "val_loss", "val_macro_f1", "grad_norm") của nhiều thí nghiệm\n    trên cùng một trục, mỗi thí nghiệm một đường, chú thích bằng exp_id.\n\n    Dùng cho ảnh figures/compare_<nhóm>.png (ví dụ compare_optimizer.png).\n    """\n    raise NotImplementedError  # TODO\n', 'results_table.py': '"""results_table.py — PSEUDO-CODE. Bạn phải tự hoàn thiện mọi hàm có `raise NotImplementedError`.\n\nNhiệm vụ: lưu kết quả từng lần chạy ra JSON, rồi điền vào experiments.xlsx từ mẫu\ntemplates/experiment_table_template.xlsx (đừng gõ tay hàng chục dòng, rất dễ sai).\n\nTên cột của sheet "Experiments" (giữ nguyên, đúng thứ tự mẫu):\n    exp_id, group, description, loss, optimizer, lr, weight_decay, batch, epochs, hidden, dropout,\n    clip_norm, precision, init, seed, step0_loss, best_val_loss, best_epoch, final_train_loss,\n    final_val_loss, val_acc, val_macro_f1, time_per_epoch_s, peak_mem_MB, diverged,\n    eval_acc, eval_macro_f1, figure_file, notes\n(các cột công thức ở cuối bảng mẫu tự tính, đừng ghi đè)\n"""\nfrom __future__ import annotations\n\nimport json\nfrom pathlib import Path\n\n\ndef save_result(result: dict, results_dir: str = "../results") -> str:\n    """Ghi result["cfg"], result["history"], result["summary"] (KHÔNG ghi best_state) ra\n    <results_dir>/<exp_id>.json. Trả về đường dẫn file. Tạo thư mục nếu chưa có."""\n    raise NotImplementedError  # TODO\n\n\ndef load_results(results_dir: str = "../results") -> list[dict]:\n    """Đọc mọi file *.json trong results_dir, trả về danh sách dict (sắp theo exp_id)."""\n    raise NotImplementedError  # TODO\n\n\ndef to_row(result: dict, eval_scores: dict | None = None, notes: str = "") -> dict:\n    """Biến một kết quả thành một dòng của bảng: gộp cfg + summary (+ eval_acc, eval_macro_f1 nếu có)\n    + figure_file = f"figures/{exp_id}.png". Khoá phải trùng tên cột ở đầu file.\n    Chỉ truyền eval_scores cho baseline và cấu hình cuối cùng."""\n    raise NotImplementedError  # TODO\n\n\ndef write_xlsx(rows: list[dict], template_path: str, out_path: str) -> None:\n    """Điền các dòng vào sheet "Experiments" của mẫu, từ dòng 2 trở xuống, rồi lưu thành out_path.\n\n    Các bước (openpyxl):\n      1. wb = openpyxl.load_workbook(template_path)   # KHÔNG dùng data_only=True (sẽ mất công thức)\n      2. ws = wb["Experiments"]; đọc tiêu đề dòng 1 để biết cột nào ứng với khoá nào\n      3. với mỗi row: ghi giá trị vào đúng cột; BỎ QUA các cột công thức (step0_gap_vs_lnC, gap_val_minus_train,\n         delta_val_f1_vs_base, beyond_noise)\n      4. wb.save(out_path)\n    Sau khi lưu, mở file bằng Excel/LibreOffice để các công thức tính lại.\n    """\n    raise NotImplementedError  # TODO\n', 'train.py': '"""train.py — PSEUDO-CODE. Bạn phải tự hoàn thiện mọi hàm có `raise NotImplementedError`.\n\nGồm: đặt seed, đánh giá, vòng huấn luyện `run_experiment(cfg, data)`, dự đoán và ghi file nộp.\nMọi thí nghiệm chỉ là *đổi dict cfg* rồi gọi lại run_experiment (xem GUIDE, Part 2).\n\nMọi chỉ số (loss, accuracy, macro-F1) dùng cùng định nghĩa với scripts/evaluate.py.\n"""\nfrom __future__ import annotations\n\nimport time\nimport random\n\nimport numpy as np\nimport torch\nimport torch.nn.functional as F\n\nfrom data import iterate_batches\nfrom model import MLP, EXPECTED_PARAMS, count_params\nfrom optimizer import build_optimizer, clip_gradients\n\n# Cấu hình mặc định = BASELINE (M-base). `lr` do bạn tự chọn bằng val rồi điền vào.\nDEFAULT_CFG = dict(\n    exp_id="base-s1", group="baseline", description="Baseline M-base",\n    loss="ce",                 # "ce" | "mse"\n    optimizer="sgd_momentum",  # "sgd" | "sgd_momentum" | "adam" | "adamw"\n    lr=None,                   # TODO: chọn bằng val, không dùng eval\n    weight_decay=0.0, momentum=0.9,\n    batch=512, epochs=20,\n    hidden=(256, 128), dropout=0.0, init="he",\n    clip_norm=None,            # None = không clip; hoặc số, ví dụ 1.0\n    precision="fp32",          # "fp32" | "fp16" | "bf16"\n    seed=1,\n)\n\n\ndef set_seed(seed: int) -> None:\n    """Đặt seed cho random, numpy, torch (và torch.cuda nếu có)."""\n    random.seed(seed)\n    np.random.seed(seed)\n    torch.manual_seed(seed)\n    if torch.cuda.is_available():\n        torch.cuda.manual_seed_all(seed)\n        torch.backends.cudnn.benchmark = False\n        torch.backends.cudnn.deterministic = True\n\n\ndef macro_f1_from_confusion(cm: np.ndarray) -> float:\n    """macro-F1 = trung bình cộng F1 của 7 lớp; F1_c = 2PR/(P+R), bằng 0 nếu P+R = 0.\n\n    cm: ma trận nhầm lẫn (7, 7), hàng = nhãn thật, cột = dự đoán.\n    """\n    raise NotImplementedError  # TODO\n\n\n@torch.no_grad()\ndef predict(model, X, batch_size: int = 8192) -> torch.Tensor:\n    """Trả về nhãn dự đoán int64 (N,) = argmax của logits.\n\n    Các bước: model.eval(); duyệt X theo từng lô (không cần xáo); gom argmax(dim=1); torch.cat.\n    """\n    raise NotImplementedError  # TODO\n\n\n@torch.no_grad()\ndef evaluate(model, X, y, loss_name: str = "ce", batch_size: int = 8192) -> dict:\n    """Trả về dict(loss, acc, macro_f1) ở chế độ eval() (dropout tắt) và no_grad.\n\n    Các bước:\n      1. model.eval()\n      2. tính logits theo từng lô; cộng dồn tổng loss (reduction="sum") rồi chia N cuối cùng\n      3. pred = argmax; acc = (pred == y).mean()\n      4. dựng ma trận nhầm lẫn 7x7 -> macro_f1_from_confusion\n    Dùng hàm này cho: train loss (trên toàn bộ hoặc một tập con CỐ ĐỊNH của train), val, và eval cuối cùng.\n    """\n    raise NotImplementedError  # TODO\n\n\ndef compute_loss(logits, y, loss_name: str):\n    """"ce"  : cross-entropy nhận logit thô và nhãn int64 (F.cross_entropy).\n       "mse" : MSE giữa logit và one-hot của y (ghi rõ bạn lấy trung bình thế nào).\n    """\n    raise NotImplementedError  # TODO\n\n\ndef run_experiment(cfg: dict, data: dict) -> dict:\n    """Huấn luyện một cấu hình và trả về lịch sử + tóm tắt.\n\n    Args:\n        cfg : dict cấu hình (xem DEFAULT_CFG)\n        data: kết quả của data.prepare_data (tensor X_tr, y_tr, X_val, y_val, X_eval, y_eval trên device)\n\n    Trả về dict:\n        {"cfg": cfg,\n         "history": {"epoch": [...], "train_loss": [...], "val_loss": [...], "val_acc": [...],\n                     "val_macro_f1": [...], "grad_norm": [...], "epoch_time_s": [...]},\n         "summary": {"step0_loss", "best_val_loss", "best_epoch", "final_train_loss", "final_val_loss",\n                     "val_acc", "val_macro_f1", "time_per_epoch_s", "peak_mem_MB", "diverged"},\n         "best_state": state_dict của epoch có val_loss thấp nhất (giữ trong RAM để dự đoán eval)}\n    (tên khoá của summary trùng tên cột trong experiments.xlsx)\n\n    Các bước:\n      0. set_seed(cfg["seed"]); tạo model = MLP(...), assert count_params(model) == EXPECTED_PARAMS[hidden]\n         chuyển model lên device; tạo optimizer = build_optimizer(...)\n         nếu precision == "fp16": scaler = torch.amp.GradScaler(...)\n      1. step0_loss = evaluate(model, X_val, y_val)["loss"]   # TRƯỚC bước cập nhật đầu tiên; kỳ vọng ≈ ln 7\n      2. for epoch in 1..epochs:\n           model.train()\n           for xb, yb in iterate_batches(X_tr, y_tr, cfg["batch"], generator):\n               with torch.autocast(...)  nếu precision != "fp32":   # chỉ bọc forward + loss\n                   logits = model(xb); loss = compute_loss(logits, yb, cfg["loss"])\n               optimizer.zero_grad(set_to_none=True)\n               backward (qua scaler nếu fp16)\n               nếu fp16 và có clip: scaler.unscale_(optimizer)  TRƯỚC khi clip\n               gn = clip_gradients(model.parameters(), cfg["clip_norm"])   # chuẩn TRƯỚC khi cắt; ghi lại\n               bước cập nhật (scaler.step(optimizer); scaler.update() nếu fp16, ngược lại optimizer.step())\n               nếu loss là NaN/inf: đặt diverged=True và dừng sớm, ĐỪNG để notebook treo\n           cuối epoch (dùng evaluate, chế độ eval):\n               train_loss trên toàn bộ train (hoặc 1 tập con CỐ ĐỊNH ~50 000 mẫu), val_loss/val_acc/val_macro_f1\n               grad_norm trung bình của epoch; thời gian epoch (torch.cuda.synchronize() nếu dùng GPU)\n               nếu val_loss tốt nhất từ trước tới giờ: lưu best_state (bản sao state_dict) và best_epoch\n      3. tổng hợp summary tại best_epoch (val_acc, val_macro_f1 lấy ở best_epoch); peak_mem_MB nếu có GPU\n    TUYỆT ĐỐI không đưa X_eval vào hàm này để chọn epoch/cấu hình. Chỉ dùng val.\n    """\n    raise NotImplementedError  # TODO\n\n\ndef write_predictions(row_id, preds, path: str) -> None:\n    """Ghi file nộp cho scripts/evaluate.py: CSV có tiêu đề `row_id,pred`.\n\n    row_id : mảng row_id của tập eval (data["eval_row_id"])\n    preds  : nhãn dự đoán int64 0..6 (cùng thứ tự với row_id)\n    Phải đủ mọi dòng của tập eval, mỗi row_id đúng một lần.\n    """\n    raise NotImplementedError  # TODO\n\n\ndef final_eval(cfg: dict, result: dict, data: dict, pred_path: str) -> None:\n    """Dùng MỘT LẦN cho cấu hình cuối cùng (và baseline): nạp best_state, dự đoán eval, ghi predictions.\n\n    Các bước:\n      1. model = MLP(...); model.load_state_dict(result["best_state"]); lên device\n      2. preds = predict(model, data["X_eval"])  # fp32, eval mode\n      3. write_predictions(data["eval_row_id"], preds.cpu().numpy(), pred_path)\n      4. chạy `python scripts/evaluate.py --pred <pred_path>` và ghi kết quả vào bảng/báo cáo\n    """\n    raise NotImplementedError  # TODO\n'}
for name, source in sources.items():
    (dest / name).write_text(source, encoding="utf-8")
os.chdir(root)


# Lab Day 1 — Notebook (PSEUDO-CODE)

> **Đây là khung pseudo-code.** Mọi ô có `# TODO` là phần bạn phải tự viết. Hãy giữ cấu trúc mục (Part 0–4) và
> viết dự đoán/nhận xét bằng chữ của bạn. Đọc `README.md`, `GUIDE.md`, `RUBRIC.md` trước.
>
> **Quy trình làm bài**
> 1. `git pull` repo, tạo thư mục nộp: copy cả thư mục `code/` vào `submission_<MSSV>/code/`.
> 2. Mở `submission_<MSSV>/code/lab.ipynb` và làm việc **tại đó** (nên `REPO_ROOT = "../.."` đúng và ảnh/kết quả ghi vào `submission_<MSSV>/figures`, `results`).
> 3. Ở Colab: chạy ô clone repo bên dưới, rồi làm tương tự (đặt `REPO_ROOT` cho đúng).
> 4. Cuối cùng: `Restart & Run All` phải chạy hết không lỗi và giữ nguyên output.

In [ ]:
# ===== Cấu hình đường dẫn và môi trường =====
import os, sys, json, time, subprocess
from pathlib import Path
import numpy as np
import torch

# Chạy từ repo hoặc một thư mục con của repo; trên Colab cd vào repo trước.
REPO_ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
                  if (p / "data/covtype.csv.gz").is_file()
                  and (p / "scripts/split_data.py").is_file()), None)
if REPO_ROOT is None:
    raise FileNotFoundError("Hãy chuyển working directory vào repo trước khi chạy.")
CODE_DIR = REPO_ROOT / "submission_2A202602853/code"
OUT_DIR = str(CODE_DIR.parent)
os.chdir(CODE_DIR)
if str(CODE_DIR) not in sys.path:
    sys.path.insert(0, str(CODE_DIR))
REPO_ROOT = str(REPO_ROOT)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("PyTorch:", torch.__version__)
print("Device:", device)
print("GPU:", torch.cuda.get_device_name(0) if device == "cuda" else "Không có CUDA; chạy CPU")
print("REPO_ROOT:", REPO_ROOT)
print("OUT_DIR:", OUT_DIR)
for folder in ("figures", "results"):
    Path(OUT_DIR, folder).mkdir(parents=True, exist_ok=True)

from data import prepare_data, load_split, make_val_split, iterate_batches
from model import MLP, EXPECTED_PARAMS, count_params, init_weights, activation_stats
from optimizer import build_optimizer, clip_gradients
from train import DEFAULT_CFG, set_seed, evaluate, predict, run_experiment, final_eval
from plots import plot_run, plot_compare
from results_table import save_result, load_results, to_row, write_xlsx
set_seed(42)


## Part 0 — Dữ liệu
Chia sẵn bằng metadata: chạy `scripts/split_data.py` từ thư mục gốc repo (tạo `data/processed/train.npz`, `eval.npz`).
Sau đó tách validation từ train, chuẩn hoá bằng thống kê của train, đưa lên device.

In [ ]:
# ===== Part 0: chia dữ liệu, chuẩn hoá và kiểm tra =====
completed = subprocess.run(
    [sys.executable, "scripts/split_data.py"], cwd=REPO_ROOT,
    check=True, capture_output=True, text=True, encoding="utf-8",
    env={**os.environ, "PYTHONUTF8": "1"},
)
print(completed.stdout)
data = prepare_data(device, val_fraction=0.2, seed=42,
                    processed_dir=f"{REPO_ROOT}/data/processed")

expected = {"tr": 371_847, "val": 92_962, "eval": 116_203}
ratios = []
for name, n in expected.items():
    X, y = data[f"X_{name}"], data[f"y_{name}"]
    assert X.shape == (n, 54) and X.dtype == torch.float32
    assert y.shape == (n,) and y.dtype == torch.int64
    assert X.device.type == device and y.device.type == device
    assert torch.isfinite(X).all().item()
    assert ((y >= 0) & (y < 7)).all().item()
    assert ((X[:, 10:] == 0) | (X[:, 10:] == 1)).all().item()
    ratios.append(np.bincount(y.cpu().numpy(), minlength=7) / n)
assert expected["tr"] + expected["val"] == 464_809
print("\nPhân bố nhãn (%), thứ tự lớp 0..6:")
for name, ratio in zip(expected, ratios):
    print(name, np.round(ratio * 100, 4))
np.testing.assert_allclose(ratios[0], ratios[1], atol=1e-4, rtol=0)
np.testing.assert_allclose(ratios[0], ratios[2], atol=1e-4, rtol=0)

# Đối chiếu dữ liệu gốc để kiểm tra cột nhị phân và thống kê chỉ lấy từ train.
X_full, y_full, X_eval_raw, _, ids = load_split(f"{REPO_ROOT}/data/processed")
X_raw, _, X_val_raw, _ = make_val_split(X_full, y_full, 0.2, 42)
np.testing.assert_allclose(data["mean"], X_raw[:, :10].mean(0, dtype=np.float64))
raw_std = X_raw[:, :10].std(0, dtype=np.float64)
np.testing.assert_allclose(data["std"], np.where(raw_std == 0, 1, raw_std))
for name, raw in (("tr", X_raw), ("val", X_val_raw), ("eval", X_eval_raw)):
    np.testing.assert_array_equal(data[f"X_{name}"].cpu().numpy()[:, 10:], raw[:, 10:])
np.testing.assert_array_equal(data["eval_row_id"], ids)
numeric = data["X_tr"].cpu().numpy()[:, :10]
mean = numeric.mean(0, dtype=np.float64)
std = numeric.std(0, dtype=np.float64)
print("\nMean 10 cột train sau chuẩn hoá:", np.round(mean, 7))
print("Std 10 cột train sau chuẩn hoá:", np.round(std, 7))
np.testing.assert_allclose(mean, 0, atol=1e-5)
np.testing.assert_allclose(std, 1, atol=1e-5)
majority = int(data["y_tr"].bincount(minlength=7).argmax().item())
majority_acc = (data["y_val"] == majority).double().mean().item()
print(f"Majority class = {majority}; val accuracy = {majority_acc:.6f}")

# Xáo bằng generator cùng device; giữ batch cuối, không bỏ mẫu.
g = torch.Generator(device=device).manual_seed(42)
seen, batches, last_batch = 0, 0, 0
for xb, yb in iterate_batches(data["X_tr"], data["y_tr"], 512, generator=g):
    assert len(xb) == len(yb)
    seen += len(xb)
    batches += 1
    last_batch = len(xb)
assert seen == expected["tr"]
print(f"Batch check: {seen} mẫu, {batches} batch, batch cuối {last_batch} mẫu")
print("PASS: tất cả kiểm tra Part 0; eval không tham gia fit chuẩn hoá/chọn cấu hình.")
del X_full, y_full, X_eval_raw, X_raw, X_val_raw, numeric, raw


### Nhận xét Part 0
- Train/val/eval được kiểm tra kích thước, dtype, phân bố 7 lớp và chuẩn hoá trong output phía trên.
- Mean/std chỉ fit trên 371 847 mẫu train sau khi tách validation. Nếu fit trên val/eval, thông tin của tập kiểm tra ảnh hưởng tiền xử lý và gây rò rỉ dữ liệu. Val/eval chỉ được transform bằng thống kê train.
- 44 cột nhị phân giữ nguyên; batch cuối được dùng đầy đủ. Validation luôn dùng seed 42, độc lập với seed các thí nghiệm sau.
- Tensor được đưa lên CUDA một lần khi CUDA khả dụng; nếu không, chạy CPU và in rõ device. Không dùng DataLoader.
